# 05B — Entity Canonicalization and Review

Input: the 5.1M raw GLiNER mentions from 05A.

`pipeline/s30_entities.py`:

1. keeps mentions with score ≥ 0.50 that fall inside the de-duplicated analysis blocks from 03;
2. canonical key: lowercase, punctuation and possessive removed, legal suffixes stripped (Inc, Corp, Ltd …), then a small alias table (`pipeline/data/entity_aliases.csv`: Facebook → Meta, Alphabet → Google, US → United States …);
3. display name = most frequent surface form; type = score-weighted vote over GLiNER labels;
4. the 15,000 most-documented keys are reviewed by the LLM with two short contexts each: keep/drop, type (company, technology, government institution, person, other organization), named vs. generic, and a canonical name. Renames that collide merge the entities.

Outputs in `results/entities/`: `entities.parquet`, `mentions.parquet`, `entity_stats.json`, `entity_review_llm.jsonl`.

In [1]:
import os, sys

PROJECT_DIR = "/content/drive/MyDrive/NLP/NLP_FINAL_PROJECT_Tom_Chen"
IN_COLAB = os.path.isdir("/content")
if IN_COLAB and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.environ.setdefault("PROJECT_DIR", PROJECT_DIR)
if IN_COLAB and "DEEPSEEK_API_KEY" not in os.environ:
    from google.colab import userdata
    os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY").strip()
# Path to a checkout of this repository (on Drive in Colab).
sys.path.insert(0, os.environ.get("REPO_DIR", f"{os.environ['PROJECT_DIR']}/code"))

import json
import pandas as pd
from pipeline import config

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [2]:
RUN = False  # set True to rebuild (CPU ~3 min + ~15K LLM calls, about $0.7 off-peak)
if RUN:
    from pipeline import s30_entities
    s30_entities.main()

In [3]:
stats = json.load(open(config.out("entities", "entity_stats.json")))
pd.Series({k: v for k, v in stats.items() if not isinstance(v, (dict, list))}).to_frame("value")

,value
min_score,0.5
raw_mentions,5113052.0
mentions_kept,2574792.0
keys_before_review,337754.0
reviewed,14964.0
review_dropped,3859.0
review_retyped,3778.0
final_entities,332376.0
final_entities_ge30_docs,2305.0


## Most-covered entities by type

In [4]:
ents = pd.read_parquet(config.out("entities", "entities.parquet"))
for t in ["company", "technology", "government_institution", "person", "other_organization"]:
    top = ents[ents.type == t].head(12)
    print(f"{t:>24}: " + "; ".join(f"{r.name} ({r.n_docs:,})" for r in top.itertuples()))

                 company: OpenAI (24,668); Google (19,932); Microsoft (16,140); Nvidia (9,580); Meta (8,031); Apple (6,121); Amazon (5,385); Anthropic (3,338); AMD (2,292); Tesla (2,173); IBM (2,169); Intel (2,149)
              technology: artificial intelligence (79,173); ChatGPT (21,008); Generative AI (8,409); machine learning (8,313); large language models (3,454); Gemini (2,879); GPT-4 (2,698); AI systems (2,297); iPhone (2,050); Natural Language Processing (1,963); AI agents (1,812); chatbot (1,731)
  government_institution: European Union (1,834); United States Congress (1,174); White House (1,151); European Commission (780); United States government (660); Federal Trade Commission (646); Trump administration (592); Biden administration (583); Federal Reserve (442); United States Senate (393); European Parliament (363); FDA (359)
                  person: Sam Altman (6,324); Elon Musk (5,385); Donald Trump (3,263); Joe Biden (1,728); Jensen Huang (1,670); Mark Zuckerberg (1,497

## What the review changed

Dropped candidates are mostly vague phrases and extraction fragments; merges collapse spelling variants.

In [5]:
rev = pd.DataFrame([{"key": r["id"], **r["label"]} for r in
                    map(json.loads, open(config.out("entities", "entity_review_llm.jsonl"))) if r["status"] == "ok"])
print(rev.keep.value_counts().to_string())
rev[~rev.keep].head(15)

keep
True     11106
False     3859


,key,keep,type,kind,canonical_name
4,ai models,False,technology,generic,AI models
6,researchers,False,person,generic,researchers
16,users,False,other_organization,generic,users
20,companies,False,other_organization,generic,companies
22,you,False,person,generic,you
25,developers,False,person,generic,developers
30,ai tools,False,technology,generic,AI tools
35,businesses,False,other_organization,generic,businesses
37,he,False,person,named,he
39,customers,False,other_organization,generic,customers


In [6]:
ents.sort_values("n_keys_merged", ascending=False)[["name", "type", "n_docs", "n_keys_merged"]].head(15)

,name,type,n_docs,n_keys_merged
2,ChatGPT,technology,21008,21
6,Generative AI,technology,8409,17
7,machine learning,technology,8313,13
17,Gemini,technology,2879,12
18,GPT-4,technology,2698,11
25,Natural Language Processing,technology,1963,10
35,deep learning,technology,1643,10
16,Donald Trump,person,3263,10
160,Nvidia H100,technology,435,10
47,computer vision,technology,1370,9
